# 01. 모델 메모리와 컨텍스트 예산

목표: Ornith-1.5-9B 실행 전에 가중치 크기와 전체 서빙 메모리를 구분하고, 하드웨어에 맞는 시작 설정을 고른다. 이 노트북은 모델을 다운로드하지 않으며 Python 표준 라이브러리만 사용한다.

## 1. 정밀도별 가중치 크기

가중치만의 이론적 크기는 `파라미터 수 × 파라미터당 byte`다. 실제 파일에는 metadata가 추가되고 quantization은 group scale 같은 보조 값을 사용하므로 결과는 근사치다.

In [ ]:
def weight_gib(parameters_billions: float, bits_per_weight: float) -> float:
    """가중치만의 이론적 GiB를 계산한다."""
    total_bytes = parameters_billions * 1_000_000_000 * bits_per_weight / 8
    return total_bytes / (1024 ** 3)


for params in (9, 10):
    estimates = {
        name: round(weight_gib(params, bits), 2)
        for name, bits in {"BF16": 16, "INT8": 8, "4-bit": 4}.items()
    }
    print(f"{params}B 이론값:", estimates)

# 공식 본문의 약 19 GB는 decimal GB 표현과 model metadata를 포함할 수 있다.
assert weight_gib(10, 16) < 19

## 2. 안전한 시작 profile 선택

정확한 KV cache는 architecture, dtype, batch, runtime에 따라 달라진다. 여기서는 검증되지 않은 VRAM 공식을 만들지 않고 운영 profile을 선택하는 보수적 규칙을 연습한다.

In [ ]:
def choose_start_profile(vram_gib: int, system_ram_gib: int) -> dict:
    """첫 성공 경로를 위한 보수적 profile을 반환한다."""
    if vram_gib >= 80:
        return {"runtime": "vLLM/SGLang", "precision": "BF16", "context": 262_144}
    if vram_gib >= 24:
        return {"runtime": "vLLM 또는 GGUF", "precision": "BF16/quantized 비교", "context": 16_384}
    if vram_gib >= 8 or system_ram_gib >= 16:
        return {"runtime": "Ollama/llama.cpp", "precision": "quantized", "context": 8_192}
    return {"runtime": "remote test endpoint 또는 dry-run", "precision": "N/A", "context": 4_096}


machines = [(80, 128), (24, 64), (8, 32), (0, 8)]
for vram, ram in machines:
    print(f"VRAM={vram}GiB RAM={ram}GiB -> {choose_start_profile(vram, ram)}")

assert choose_start_profile(8, 32)["context"] == 8_192

## 3. 실행 전 체크리스트

1. 사용 가능한 VRAM과 system RAM을 기록한다.
2. disk에 모델과 cache를 위한 여유 공간이 있는지 확인한다.
3. 첫 실행은 8K~16K context와 batch 1로 시작한다.
4. peak memory, first-token latency, token throughput을 기록한다.
5. context를 단계적으로 늘리고 OOM이 나면 마지막 성공값으로 돌아간다.

다음 노트북에서는 실제 서버를 켜지 않고도 API request와 평가 contract를 먼저 만든다.